# 08 — Matrix Computation (GEMV & GEMM)

## Background

Matrix multiplication (GEMM) is the most compute-intensive operation in deep learning: linear layers, attention projections, and feed-forward networks all depend on it. Modern GPUs have dedicated matrix-multiply hardware: AMD RDNA3/RDNA4 GPUs expose this via WMMA instructions (`v_wmma_f32_16x16x16_f16`), delivering up to ~190 TFLOPS fp16 on the RX 7900 XTX, but reaching peak performance requires:

- **WMMA (Wave Matrix Multiply-Accumulate)**: processes 16×16 matrix blocks via `v_wmma_f32_16x16x16_f16`; significantly higher throughput than scalar Shader Processors (VALU)
- **Shared Memory**: on-chip SRAM, far higher bandwidth than global GPU memory (GDDR6)
- **Software Pipelining**: overlaps global-memory loads with WMMA compute, hiding memory latency

## Problem Definition

**GEMM**: $C[i,j] = \sum_k A[i,k] \times B[k,j]$, $A\in\mathbb{R}^{M\times K}$, $B\in\mathbb{R}^{K\times N}$

| Implementation | A/B buffer | K loop |
|---------------|-----------|--------|
| Triton | automatic | `for k in range(0, K, BK)` |
| TileLang naive | `alloc_fragment` (registers) | `T.Serial` |
| TileLang opt | `alloc_shared` (shared memory) | `T.Pipelined(num_stages=3)` |

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
M, N, K = 4096, 4096, 4096
A = torch.randn(M, K, dtype=torch.float16, device="cuda")
B = torch.randn(K, N, dtype=torch.float16, device="cuda")

def ref_gemm(A, B): return torch.matmul(A, B)

C_ref = ref_gemm(A, B)
print(f"A: {A.shape} × B: {B.shape} → C: {C_ref.shape}")
print(f"Total FLOPs: {2*M*N*K:.2e}")

## Triton Implementation

The classic Triton GEMM: 2D block grid over the output matrix, inner K loop loads A/B tiles, `tl.dot` triggers WMMA MMA. `allow_tf32=True` enables TF32 acceleration on supported GPUs.

In [ ]:
@triton.jit
def _matmul_kernel(A_ptr, B_ptr, C_ptr, M, N, K,
                   BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows  = pid_m * BM + tl.arange(0, BM)
    cols  = pid_n * BN + tl.arange(0, BN)
    acc   = tl.zeros([BM, BN], dtype=tl.float32)
    for k in range(0, K, BK):
        k_range = k + tl.arange(0, BK)
        a = tl.load(A_ptr + rows[:, None] * K + k_range[None, :],
                    mask=(rows[:, None] < M) & (k_range[None, :] < K), other=0.0)
        b = tl.load(B_ptr + k_range[:, None] * N + cols[None, :],
                    mask=(k_range[:, None] < K) & (cols[None, :] < N), other=0.0)
        acc = acc + tl.dot(a, b, allow_tf32=True)
    tl.store(C_ptr + rows[:, None] * N + cols[None, :], acc.to(tl.float16),
             mask=(rows[:, None] < M) & (cols[None, :] < N))

BM_t, BN_t, BK_t = 128, 128, 64

def triton_gemm(A, B):
    M_, K_ = A.shape;  _, N_ = B.shape
    C = torch.empty(M_, N_, dtype=torch.float16, device=A.device)
    grid = (triton.cdiv(M_, BM_t), triton.cdiv(N_, BN_t))
    _matmul_kernel[grid](A, B, C, M_, N_, K_, BM=BM_t, BN=BN_t, BK=BK_t)
    return C

C_tri = triton_gemm(A, B)
ok = torch.allclose(C_ref.half(), C_tri.half(), atol=1.0)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(C_ref - C_tri)).item():.4f}")

## TileLang Implementation

Two variants:

**Naive** (`alloc_fragment` + `T.Serial`) — register-only accumulation, no data reuse, slow.

**WMMA** — uses the RDNA3 `v_wmma_f32_16x16x16_f16` instruction (warp-size=32) via `WMMAIntrinEmitter`:
- Shared memory with `make_mfma_swizzle_layout` to eliminate bank conflicts
- `T.use_swizzle(panel_size=10)` for L2 cache swizzling across blocks
- `WMMAIntrinEmitter.ldmatrix_a/b` load A/B fragments from shared → registers
- `wmma_e.wmma(A_l, B_l, C_l)` emits the `v_wmma` instruction
- `wmma_e.stmatrix` writes C back with the correct WMMA output layout

Best config on RX 7900 XTX: `brw=2, bcw=2, wrt=64, wct=64, chunk=64` → **~85 TFLOPS** (vs rocBLAS ~90 TFLOPS, Triton ~68 TFLOPS).

> **Note**: B must be passed in transposed form `(N×K)` for the `b_transposed=True` WMMA layout; `B_t = B.T.contiguous()` is prepared once before benchmarking.

In [ ]:
from tilelang.rocm.intrinsics.wmma_macro_generator import WMMAIntrinEmitter
from tilelang.rocm.intrinsics import make_mfma_swizzle_layout as make_swizzle_layout
from tilelang.transform import simplify_prim_func
from tilelang.utils import determine_target

# ── Naive SIMT GEMM ─────────────────────────────────────────────────────────
BM, BN, BK = 128, 128, 32

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_gemm_naive(A, B, BLOCK_M: int, BLOCK_N: int, BLOCK_K: int):
    M_, N_, K_ = T.const("M, N, K")
    dtype, acc = T.float16, T.float32
    A: T.Tensor((M_, K_), dtype);  B: T.Tensor((K_, N_), dtype)
    C = T.empty((M_, N_), dtype)
    with T.Kernel(T.ceildiv(N_, BLOCK_N), T.ceildiv(M_, BLOCK_M), threads=128) as (pid_n, pid_m):
        A_l = T.alloc_fragment((BLOCK_M, BLOCK_K), dtype)
        B_l = T.alloc_fragment((BLOCK_K, BLOCK_N), dtype)
        C_l = T.alloc_fragment((BLOCK_M, BLOCK_N), acc)
        T.clear(C_l)
        for k in T.Serial(K_ // BLOCK_K):
            T.copy(A[pid_m * BLOCK_M, k * BLOCK_K], A_l)
            T.copy(B[k * BLOCK_K, pid_n * BLOCK_N], B_l)
            T.gemm(A_l, B_l, C_l)
        T.copy(C_l, C[pid_m * BLOCK_M, pid_n * BLOCK_N])
    return C

k_naive = tl_gemm_naive.compile(M=M, N=N, K=K, BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK)
ok = torch.allclose(C_ref.half(), k_naive(A.clone(), B.clone()).half(), atol=1.0)
print(f"  TileLang naive   : {'✅ PASSED' if ok else '❌ FAILED'}")

# ── Optimised WMMA GEMM ──────────────────────────────────────────────────────
# All three archs use WMMA (v_wmma_f32_16x16x16_f16, warp_size=32).
# gfx1100 (RX 7900 XTX): brw=bcw=2, wrt=wct=64, chunk=64, panel=10 → ~86 TFLOPS
# gfx1201 (R9700):        brw=bcw=2, wrt=wct=64, chunk=64, panel=10 → ~118 TFLOPS
# gfx1151 (Radeon 8060S): brw=bcw=2, wrt=wct=64, chunk=64, panel=4  → ~28 TFLOPS
#   Sweep over 1260 configs shows wrt=wct=64 beats wrt=32; larger tile improves
#   arithmetic intensity. panel=4 (smaller L2 swizzle) is optimal for 128-block grid.
arch = torch.cuda.get_device_properties(0).gcnArchName
if arch.startswith("gfx1151"):
    brw, bcw, wrt, wct, chunk, panel = 2, 2, 64, 64, 64, 4
else:
    brw, bcw, wrt, wct, chunk, panel = 2, 2, 64, 64, 64, 10

B_t = B.T.contiguous()   # (K,N) → (N,K), required by b_transposed=True

target = determine_target("auto", return_object=True)
block_M_w = brw * wrt; block_N_w = bcw * wct; block_K_w = chunk

wmma_e = WMMAIntrinEmitter(
    a_dtype=T.float16, b_dtype=T.float16, accum_dtype=T.float32,
    a_transposed=False, b_transposed=True,
    block_row_warps=brw, block_col_warps=bcw,
    warp_row_tiles=wrt, warp_col_tiles=wct,
    chunk=chunk, k_pack=1, target=target,
)
threads_w  = wmma_e.threads
la, lb, lc = wmma_e.local_size_a, wmma_e.local_size_b, wmma_e.local_size_out
wr, wc     = wmma_e.warp_rows, wmma_e.warp_cols

@simplify_prim_func
def _wmma_kernel():
    @T.prim_func
    def main(
        A_in: T.Tensor((M, K), T.float16),
        B_in: T.Tensor((N, K), T.float16),
        C_out: T.Tensor((M, N), T.float16),
    ):
        with T.Kernel(T.ceildiv(N, block_N_w), T.ceildiv(M, block_M_w), threads=threads_w) as (bx, by):
            A_s = T.alloc_shared((block_M_w, block_K_w), T.float16)
            B_s = T.alloc_shared((block_N_w, block_K_w), T.float16)
            A_l = T.alloc_fragment((wr * la,), T.float16)
            B_l = T.alloc_fragment((wc * lb,), T.float16)
            C_l = T.alloc_fragment((wr * wc * lc,), T.float32)
            T.annotate_layout({
                A_s: make_swizzle_layout(A_s),
                B_s: make_swizzle_layout(B_s),
            })
            T.use_swizzle(panel_size=panel)
            T.clear(C_l)
            for ko in T.Pipelined(K // block_K_w, num_stages=0):
                T.copy(A_in[by * block_M_w, ko * block_K_w], A_s)
                T.copy(B_in[bx * block_N_w, ko * block_K_w], B_s)
                for ki in T.serial(0, block_K_w // 16):
                    wmma_e.ldmatrix_a(A_l, A_s, ki)
                    wmma_e.ldmatrix_b(B_l, B_s, ki)
                    wmma_e.wmma(A_l, B_l, C_l)
            wmma_e.stmatrix(C_l, C_out, pid_m=by, pid_n=bx)
    return main

k_wmma = tilelang.compile(_wmma_kernel())

C_wmma = torch.zeros(M, N, dtype=torch.float16, device="cuda")
k_wmma(A, B_t, C_wmma)
ok = torch.allclose(C_ref.half(), C_wmma, atol=1.0)
print(f"  TileLang WMMA    : {'✅ PASSED' if ok else '❌ FAILED'}  (arch={arch}, wrt={wrt}, panel={panel})")

## Performance Comparison

Dual chart: latency (ms) and TFLOPS, showing each implementation's distance from peak hardware throughput.

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 100

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

flops = 2 * M * N * K

_C_bench = torch.zeros(M, N, dtype=torch.float16, device="cuda")  # pre-allocated, avoids per-call allocation overhead
def run_wmma(a, b_t):
    k_wmma(a, b_t, _C_bench)
    return _C_bench

results = {
    "PyTorch\n(rocBLAS)": bench(ref_gemm,        [A, B]),
    "Triton":             bench(triton_gemm,      [A, B]),
    "TileLang\nNaive":   bench(k_naive,          [A, B]),
    "TileLang\nWMMA":    bench(run_wmma,         [A, B_t]),
}

for name, ms in results.items():
    tf = flops / ms / 1e9
    print(f"  {name.replace(chr(10),' '):22s}: {ms:.4f} ms  ({tf:.1f} TFLOPS)")

colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]
labels = list(results.keys())
ms_vals = list(results.values())
tf_vals = [flops / ms / 1e9 for ms in ms_vals]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bars = ax.bar(labels, ms_vals, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, ms_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(ms_vals)*0.01,
            f"{val:.2f} ms", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Latency (ms)"); ax.set_title(f"GEMM Latency  (M=N=K={M}, float16, {arch})")
ax.set_ylim(0, max(ms_vals)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bars = ax.bar(labels, tf_vals, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, tf_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(tf_vals)*0.01,
            f"{val:.1f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Throughput (TFLOPS)"); ax.set_title(f"GEMM Throughput  (M=N=K={M}, float16, {arch})")
ax.set_ylim(0, max(tf_vals)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()